In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, trim

In [0]:
df = spark.table("workspace.bronze.products")


In [0]:
RENAME_MAP = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}
for old_name, new_name in RENAME_MAP.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

In [0]:
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    df = df.withColumn(col_name, trim(col(col_name)))

In [0]:
df = df.filter(col("product_id").isNotNull()) \
       .fillna({"product_category_name": "other"})

In [0]:
df = (
    df
    # Normalising
    .withColumn("product_category_name", 
                F.lower(F.regexp_replace(col("product_category_name"), " ", "_")))
    # Casting 
    .withColumn("product_category_name", F.lower(col("product_category_name")))
    .withColumn("product_name_length", F.when(col("product_name_length").cast(IntegerType()).isNotNull(), col("product_name_length").cast(IntegerType())).otherwise(0))
    .withColumn("product_description_length", F.when(col("product_description_length").cast(IntegerType()).isNotNull(), col("product_description_length").cast(IntegerType())).otherwise(0))
    .withColumn("product_photos_qty", F.when(col("product_photos_qty").cast(IntegerType()).isNotNull(), col("product_photos_qty").cast(IntegerType())).otherwise(0))
    .withColumn("product_weight_g", F.when(col("product_weight_g").cast(DoubleType()).isNotNull(), col("product_weight_g").cast(DoubleType())).otherwise(0.0))
    .withColumn("product_length_cm", F.when(col("product_length_cm").cast(DoubleType()).isNotNull(), col("product_length_cm").cast(DoubleType())).otherwise(0.0))
    .withColumn("product_height_cm", F.when(col("product_height_cm").cast(DoubleType()).isNotNull(), col("product_height_cm").cast(DoubleType())).otherwise(0.0))
    .withColumn("product_width_cm", F.when(col("product_width_cm").cast(DoubleType()).isNotNull(), col("product_width_cm").cast(DoubleType())).otherwise(0.0))
)


In [0]:
df = df.dropDuplicates(["product_id"])

In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())

In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.products")